In [1]:
import torch
import tiktoken
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from tqdm import tqdm
from collections import Counter

In [2]:
from modules.sampling import *
from modules.structure import *
from modules.training import *

In [3]:
import urllib.request
url = (
"https://raw.githubusercontent.com/rasbt/"
"LLMs-from-scratch/main/ch05/"
"01_main-chapter-code/gpt_download.py"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

('gpt_download.py', <http.client.HTTPMessage at 0x713234b11d30>)

In [4]:
BASE_CONFIG = {
    "vocab_size": 50257, # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 1024, # Embedding dimension
    "n_heads": 16, # Number of attention heads
    "n_layers": 24, # Number of layers
    "drop_rate": 0.05, # Dropout rate
    "qkv_bias": True # Query-Key-Value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

In [5]:
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(
model_size=model_size,
models_dir="gpt2"
)

I0000 00:00:1776793106.658736   19012 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776793106.688836   19012 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776793107.251698   19012 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


File already exists and is up-to-date: gpt2/355M/checkpoint
File already exists and is up-to-date: gpt2/355M/encoder.json
File already exists and is up-to-date: gpt2/355M/hparams.json
File already exists and is up-to-date: gpt2/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/355M/model.ckpt.index
File already exists and is up-to-date: gpt2/355M/model.ckpt.meta
File already exists and is up-to-date: gpt2/355M/vocab.bpe


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
tokenizer = tiktoken.get_encoding("gpt2")

In [8]:
semicolon_id = tokenizer.encode(";")[0]

In [9]:
model = GPTModel(BASE_CONFIG)
model.eval()

load_weights_into_gpt(model, params)
model.to(device)

torch.manual_seed(123)
token_ids = generate(
model=model,
idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
max_new_tokens=25,
context_size=BASE_CONFIG["context_length"],
top_k=50,
temperature=1.5,
eos_id=semicolon_id
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you as far as the natural capacity is capable," the lawyer wrote, "which permits extraordinary actions." "That includes (dressing


In [10]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

Total trainable parameters before: 406,286,336


In [11]:
for param in model.parameters():
    param.requires_grad = False

In [12]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after: {total_params:,}")

Total trainable parameters after: 0


In [13]:
replace_linear_with_lora(model, rank=16, alpha=16)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")

Total trainable LoRA parameters: 7,898,384


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.05, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): LinearWithLoRA(
          (linear): Linear(in_features=1024, out_features=1024, bias=True)
          (lora): LoRALayer()
        )
        (W_key): LinearWithLoRA(
          (linear): Linear(in_features=1024, out_features=1024, bias=True)
          (lora): LoRALayer()
        )
        (W_value): LinearWithLoRA(
          (linear): Linear(in_features=1024, out_features=1024, bias=True)
          (lora): LoRALayer()
        )
        (out_proj): LinearWithLoRA(
          (linear): Linear(in_features=1024, out_features=1024, bias=True)
          (lora): LoRALayer()
        )
        (dropout): Dropout(p=0.05, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): LinearWithLoRA(
            (linear): Linear(in_features=10

In [15]:
class WikiSQLDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=256):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                question, sql = line.strip().split(",", 1)

                text = (
                    f"Question: {question}\n"
                    f"SQL: {sql}"
                )

                self.samples.append(text)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        text = self.samples[idx]

        token_ids = self.tokenizer.encode(text)

        token_ids = token_ids[:self.max_length]

        if len(token_ids) < self.max_length:
            token_ids += [50256] * (self.max_length - len(token_ids))

        input_ids = torch.tensor(token_ids)

        attention_mask = (input_ids != 50256).long()

        labels = input_ids.clone()

        labels[input_ids == 50256] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
train_dataset = WikiSQLDataset(
    "WikiSQL/train.csv",
    tokenizer,
    max_length=256
)

validation_dataset = WikiSQLDataset(
    "WikiSQL/validation.csv",
    tokenizer,
    max_length=256
)

test_dataset = WikiSQLDataset(
    "WikiSQL/test.csv",
    tokenizer,
    max_length=256
)

In [17]:
num_workers = 24
batch_size = 2

train_loader = DataLoader(
    train_dataset,
    batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False
)

In [18]:
'''
for param in model.parameters():
    param.requires_grad = False

for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True
    '''

'\nfor param in model.parameters():\n    param.requires_grad = False\n\nfor param in model.trf_blocks[-1].parameters():\n    param.requires_grad = True\nfor param in model.final_norm.parameters():\n    param.requires_grad = True\n    '

In [19]:
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    weight_decay=0.01
)

initial_lr = 1e-6
peak_lr = 1e-4

criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

train_model(
    model,
    train_loader,
    validation_loader,
    optimizer,
    criterion,
    device,
    initial_lr,
    peak_lr,
    num_epochs=2
)

Training: 100%|██████████| 28178/28178 [1:06:28<00:00,  7.07it/s, avg=1.81, loss=2.05] 



Epoch 1/2 | Train Loss: 1.8102 | Val Loss: 1.6963



Training: 100%|██████████| 28178/28178 [1:06:59<00:00,  7.01it/s, avg=1.66, loss=2.14] 



Epoch 2/2 | Train Loss: 1.6575 | Val Loss: 1.6607



In [23]:
prompt = """Which fruit does have the aminoacid X?
SQL:"""

output = evaluate_generation(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    device=device,
    max_new_tokens=50
)

print(output)

Which fruit does have the aminoacid X?
SQL: SELECT Amino Acid FROM table WHERE Amino Acid = x


In [21]:
torch.save(model.state_dict(), "models/modelo_04.pth")